## 데이터 로드 / 셋업

In [2]:
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

ROOT = Path.cwd()
TRAIN_PATH = ROOT / 'train_cleaned.csv'
TEST_PATH = ROOT / 'test_cleaned.csv'
OUT_DIR = ROOT / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 저장
TRAIN_OUT = OUT_DIR / 'train_eda.csv'
TEST_OUT = OUT_DIR / 'test_eda.csv'

# 날짜 계산에 사용할 날짜 컬럼
DATE_COLS = ['baseline_create_date', 'due_in_date', 'clear_date']

FEATURE_COLUMNS = [
    'business_days_late',
    'target',
    'due_weekend_flag',
    'business_day_gap',
    'amount_bin',
    'cust_allowed_pay_days_late_rate_past',
    'ratio_paid_invoices_late_past',
    'avg_days_late_paid_late_past',
    'sum_outstanding_amount_past',
    'recent_5_late_rate',
]

dtype = {
    'cust_number': 'string',
    'business_code': 'string',
    'name_customer': 'string',
    'cust_payment_terms': 'string',
    'cust_payment_terms_grp': 'string',
}

# 기존 target은 target_old로
# 새 target은 새 기준으로 다시 생성
train_raw = pd.read_csv(TRAIN_PATH, dtype=dtype).rename(columns={'target': 'target_old'})
test_raw = pd.read_csv(TEST_PATH, dtype=dtype).rename(columns={'target': 'target_old'})

train_featured = train_raw.copy()
test_featured = test_raw.copy()

# datetime
for df in [train_featured, test_featured]:
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors='raise')

# 확인
for name, df in [('train', train_featured), ('test', test_featured)]:
    print(f'{name}: rows={len(df):,}, cols={len(df.columns)}, customers={df["cust_number"].nunique():,}')
    print('target_old counts:', df['target_old'].value_counts(dropna=False).sort_index().to_dict())
    print('missing values:', int(df.isna().sum().sum()))
    display(df.head(3)) if 'display' in globals() else print(df.head(3).to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\user\\OneDrive\\바탕 화면\\invoice-delay-prediction\\notebooks\\train_cleaned.csv'

## business_days_late


In [ ]:
# 만기일과 실제 결제일 사이의 지연일을 영업일 기준으로 계산
def add_business_days_late(df: pd.DataFrame) -> None:
    # np.busday_count
    due_days = df['due_in_date'].values.astype('datetime64[D]')
    clear_days = df['clear_date'].values.astype('datetime64[D]')

    # due_in_date부터 clear_date까지 영업일
    df['business_days_late'] = np.busday_count(due_days, clear_days).astype(int)

add_business_days_late(train_featured)
add_business_days_late(test_featured)

print(train_featured['business_days_late'].describe())

count    32000.000000
mean         0.618437
std          7.928335
min        -63.000000
25%         -2.000000
50%          0.000000
75%          1.000000
max        146.000000
Name: business_days_late, dtype: float64


## target


In [ ]:
# 영업일 기준으로 5일을 초과해 늦게 결제된 경우
# business_days_late > 5 면 1 아니면 0
train_featured['target'] = (train_featured['business_days_late'] > 5).astype(int)
test_featured['target'] = (test_featured['business_days_late'] > 5).astype(int)

# 기존 target_old와 target의 분포 확인해보기
print(pd.DataFrame({
    'train_target_old': train_featured['target_old'].value_counts().sort_index(),
    'train_target': train_featured['target'].value_counts().sort_index(),
    'test_target_old': test_featured['target_old'].value_counts().sort_index(),
    'test_target': test_featured['target'].value_counts().sort_index(),
}).fillna(0).astype(int))

   train_target_old  train_target  test_target_old  test_target
0             18393         29870             4843         7567
1             13607          2130             3157          433


## due_weekend_flag


In [ ]:
# due_in_date가 토요일 또는 일요일이면 1, 평일이면 0
train_featured['due_weekend_flag'] = train_featured['due_in_date'].dt.weekday.isin([5, 6]).astype(int)
test_featured['due_weekend_flag'] = test_featured['due_in_date'].dt.weekday.isin([5, 6]).astype(int)

print('train:', train_featured['due_weekend_flag'].value_counts().sort_index().to_dict())
print('test :', test_featured['due_weekend_flag'].value_counts().sort_index().to_dict())

train: {0: 23110, 1: 8890}
test : {0: 5773, 1: 2227}


## business_day_gap


In [ ]:
# 만기일이 다음 영업일까지 얼마나 떨어져 있는가?
def add_business_day_gap(df: pd.DataFrame) -> None:
    due_weekday = df['due_in_date'].dt.weekday

    # 토요일 만기 = 2일, 일요일 만기= 1일
    df['business_day_gap'] = np.select(
        [due_weekday.eq(5), due_weekday.eq(6)],
        [2, 1],
        default=0,
    ).astype(int)

add_business_day_gap(train_featured)
add_business_day_gap(test_featured)

print('train:', train_featured['business_day_gap'].value_counts().sort_index().to_dict())
print('test :', test_featured['business_day_gap'].value_counts().sort_index().to_dict())

train: {0: 23110, 1: 3831, 2: 5059}
test : {0: 5773, 1: 960, 2: 1267}


## amount_bin


In [ ]:
# amount_in_usd를 train 데이터에서4분위 구간 분할
_, raw_amount_bins = pd.qcut(
    train_featured['amount_in_usd'],
    q=4,
    labels=False,
    retbins=True,
    duplicates='drop',
)

amount_bins = np.r_[-np.inf, raw_amount_bins[1:-1], np.inf]
amount_labels = list(range(len(amount_bins) - 1))

train_featured['amount_bin'] = pd.cut(
    train_featured['amount_in_usd'],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True,
).astype('int64')

test_featured['amount_bin'] = pd.cut(
    test_featured['amount_in_usd'],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True,
).astype('int64')

print('Amount bin edges:', amount_bins)
print('train:', train_featured['amount_bin'].value_counts().sort_index().to_dict())
print('test :', test_featured['amount_bin'].value_counts().sort_index().to_dict())

Amount bin edges: [       -inf  8.4071601   9.73095483 10.72198876         inf]
train: {0: 8000, 1: 8000, 2: 8000, 3: 8000}
test : {0: 1901, 1: 1975, 2: 2008, 3: 2116}


## cust_allowed_pay_days_late_rate_past


In [ ]:
feature = 'cust_allowed_pay_days_late_rate_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 결제 완료된 송장만
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        # 같은 고객 + 같은 Allowed_Pay_Days의 과거 연체율
        rate_by_allowed_days = paid.groupby('Allowed_Pay_Days')['target'].mean()
        train_featured.loc[current_rows.index, feature] = (
            current_rows['Allowed_Pay_Days'].map(rate_by_allowed_days).fillna(0.0).astype(float)
        )

# test는 train 전체 + 현재보다 과거인 test 이력만
test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        rate_by_allowed_days = paid.groupby('Allowed_Pay_Days')['target'].mean()
        test_featured.loc[current_rows.index, feature] = (
            current_rows['Allowed_Pay_Days'].map(rate_by_allowed_days).fillna(0.0).astype(float)
        )

print(train_featured[feature].describe())

count    32000.000000
mean         0.049603
std          0.160327
min          0.000000
25%          0.000000
50%          0.003726
75%          0.017668
max          1.000000
Name: cust_allowed_pay_days_late_rate_past, dtype: float64


## ratio_paid_invoices_late_past


In [8]:
feature = 'ratio_paid_invoices_late_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 같은 고객의 과거 결제 완료 송장 중 target 기준 연체 비율
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if not paid.empty:
            train_featured.loc[current_rows.index, feature] = float(paid['target'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if not paid.empty:
            test_featured.loc[current_rows.index, feature] = float(paid['target'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean         0.050203
std          0.159811
min          0.000000
25%          0.000000
50%          0.004625
75%          0.019608
max          1.000000
Name: ratio_paid_invoices_late_past, dtype: float64


## avg_days_late_paid_late_past


In [9]:
feature = 'avg_days_late_paid_late_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 같은 고객의 과거 연체 송장만 대상으로 평균 영업일 지연일
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid_late = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
            & (customer_history['target'].eq(1))
        ]
        if not paid_late.empty:
            train_featured.loc[current_rows.index, feature] = float(paid_late['business_days_late'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid_late = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
            & (customer_history['target'].eq(1))
        ]
        if not paid_late.empty:
            test_featured.loc[current_rows.index, feature] = float(paid_late['business_days_late'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean        13.934218
std         17.427254
min          0.000000
25%          0.000000
50%          8.000000
75%         18.000000
max         86.000000
Name: avg_days_late_paid_late_past, dtype: float64


## sum_outstanding_amount_past


In [10]:
feature = 'sum_outstanding_amount_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 아직 결제되지 않은 과거 송장의 총금액
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        outstanding = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] >= current_date)
        ]
        train_featured.loc[current_rows.index, feature] = max(float(outstanding['amount_in_usd'].sum()), 0.0)

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        outstanding = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] >= current_date)
        ]
        test_featured.loc[current_rows.index, feature] = max(float(outstanding['amount_in_usd'].sum()), 0.0)

print(train_featured[feature].describe())

count    32000.000000
mean       740.875133
std       1071.324304
min          0.000000
25%         35.262890
50%        162.661170
75%        650.793740
max       3635.495781
Name: sum_outstanding_amount_past, dtype: float64


## recent_5_late_rate


In [11]:
feature = 'recent_5_late_rate'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 최근 5건 결제 완료 송장 중 연체 비율
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)].copy()
    customer_history['_eligible_paid_date'] = customer_history[['baseline_create_date', 'clear_date']].max(axis=1)

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        recent_paid = paid.sort_values(['_eligible_paid_date', 'baseline_create_date', 'clear_date']).tail(5)
        train_featured.loc[current_rows.index, feature] = float(recent_paid['target'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)].copy()
    customer_history['_eligible_paid_date'] = customer_history[['baseline_create_date', 'clear_date']].max(axis=1)

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        recent_paid = paid.sort_values(['_eligible_paid_date', 'baseline_create_date', 'clear_date']).tail(5)
        test_featured.loc[current_rows.index, feature] = float(recent_paid['target'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean         0.041245
std          0.160203
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: recent_5_late_rate, dtype: float64


## csv 저장


In [3]:
train_featured.to_csv(TRAIN_OUT, index=False)
test_featured.to_csv(TEST_OUT, index=False)

print('Saved:', TRAIN_OUT)
print('Saved:', TEST_OUT)
print('Train output exists:', TRAIN_OUT.exists(), 'size:', TRAIN_OUT.stat().st_size)
print('Test output exists :', TEST_OUT.exists(), 'size:', TEST_OUT.stat().st_size)

NameError: name 'train_featured' is not defined

아래 코드는 train_eda를 수정하여 보완한 코드

In [10]:
train_eda_revised=pd.read_csv('../data/train_eda.csv')
test_eda_revised=pd.read_csv('../data/test_eda.csv')

In [11]:
import pandas as pd

# due_in_date 컬럼만 복사
check = train_eda_revised[['due_in_date']].copy()

# due_in_date를 날짜형으로 변환
check['due_in_date'] = pd.to_datetime(check['due_in_date'], errors='coerce')

# 요일 숫자 확인
check['weekday_num'] = check['due_in_date'].dt.weekday

# 요일 이름 확인
check['weekday_name'] = check['due_in_date'].dt.day_name()

print(check.head(20))

print(check.head(20))

   due_in_date  weekday_num weekday_name
0   2018-12-29            5     Saturday
1   2018-12-24            0       Monday
2   2019-01-14            0       Monday
3   2019-01-14            0       Monday
4   2019-01-14            0       Monday
5   2019-01-14            0       Monday
6   2019-01-14            0       Monday
7   2019-01-14            0       Monday
8   2019-01-14            0       Monday
9   2019-01-14            0       Monday
10  2019-01-14            0       Monday
11  2019-01-14            0       Monday
12  2019-01-14            0       Monday
13  2019-01-14            0       Monday
14  2019-01-14            0       Monday
15  2019-01-14            0       Monday
16  2019-01-14            0       Monday
17  2019-01-14            0       Monday
18  2019-01-14            0       Monday
19  2019-01-14            0       Monday
   due_in_date  weekday_num weekday_name
0   2018-12-29            5     Saturday
1   2018-12-24            0       Monday
2   2019-01-14  

In [12]:
# business_day_gap 컬럼 제거(영업일로 계산한 지연일과 겹치는 정보이므로 제거)

train_eda_revised.drop(columns=['business_day_gap'], inplace=True)
test_eda_revised.drop(columns=['business_day_gap'], inplace=True)

In [14]:
import numpy as np
import pandas as pd


# =========================================
# 1. 피처명 설정
# =========================================

feature = 'cust_allowed_pay_days_late_rate_past'

train_eda_revised[feature] = 0.0
test_eda_revised[feature] = 0.0


# =========================================
# 2. 날짜 컬럼 datetime 변환
# =========================================

date_cols = ['baseline_create_date', 'due_in_date', 'clear_date']

for col in date_cols:
    train_eda_revised[col] = pd.to_datetime(train_eda_revised[col], errors='coerce')
    test_eda_revised[col] = pd.to_datetime(test_eda_revised[col], errors='coerce')


# =========================================
# 3. 영업일 계산 함수
# =========================================

def business_days_between(start_dates, end_date):
    """
    start_dates부터 end_date까지의 영업일 수를 계산하는 함수이다.
    예: due_in_date부터 현재 baseline_create_date까지 몇 영업일이 지났는지 계산한다.
    """
    start_dates = pd.to_datetime(start_dates, errors='coerce')
    end_date = pd.to_datetime(end_date)

    result = pd.Series(np.nan, index=start_dates.index, dtype='float')

    valid_mask = start_dates.notna()

    if valid_mask.sum() == 0:
        return result

    start_np = start_dates.loc[valid_mask].values.astype('datetime64[D]')
    end_np = np.datetime64(end_date.date(), 'D')

    result.loc[valid_mask] = np.busday_count(start_np, end_np)

    return result


# =========================================
# 4. 기준 설정
# =========================================

# 타겟 정의와 동일하게 영업일 기준 5일 이상 지연으로 설정
LATE_BUSINESS_DAYS = 5


# =========================================
# 5. 피처 생성 함수
# =========================================

def add_allowed_pay_days_late_rate_past(current_df, history_df, feature):
    """
    현재 데이터(current_df)에 대해 고객별 + Allowed_Pay_Days별 과거 연체율을 생성한다.

    반영하는 과거 송장은 다음과 같다.

    1. 현재 송장 생성일 이전에 이미 완납된 송장
       - clear_date < 현재 baseline_create_date
       - 이 경우에는 target 값을 그대로 사용한다.

    2. 현재 송장 생성일 기준 아직 미완납이지만 이미 5영업일 이상 지연된 송장
       - clear_date가 없거나 clear_date >= 현재 baseline_create_date
       - 현재 baseline_create_date 기준 due_in_date로부터 5영업일 이상 지남
       - 이 경우에는 현재 시점에서 이미 연체 확정이므로 1로 반영한다.

    3. 아직 미완납이고 5영업일 이상 지연도 확정되지 않은 송장
       - 정상인지 연체인지 알 수 없으므로 계산에서 제외한다.
    """

    for cust, current_rows_for_customer in current_df.groupby('cust_number', sort=False):

        # 같은 고객의 전체 이력
        customer_history = history_df[history_df['cust_number'].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):

            # 현재 송장 생성일보다 과거에 생성된 송장만 사용
            prior = customer_history[
                customer_history['baseline_create_date'] < current_date
            ].copy()

            if prior.empty:
                continue

            # 1. 현재 시점 이전에 이미 완납된 송장
            paid_before_current = (
                prior['clear_date'].notna()
                & (prior['clear_date'] < current_date)
            )

            # 2. 현재 시점 기준 아직 미완납인 송장
            unpaid_as_of_current = (
                prior['clear_date'].isna()
                | (prior['clear_date'] >= current_date)
            )

            # due_in_date부터 현재 송장 생성일까지 지난 영업일 수 계산
            business_days_past_due = business_days_between(
                prior['due_in_date'],
                current_date
            )

            # 3. 미완납이지만 현재 시점에서 이미 5영업일 이상 지연 확정된 송장
            known_late_unpaid = (
                unpaid_as_of_current
                & (business_days_past_due >= LATE_BUSINESS_DAYS)
            )

            # 현재 시점에서 연체 여부를 알 수 있는 송장만 사용
            known_history = prior[
                paid_before_current | known_late_unpaid
            ].copy()

            if known_history.empty:
                continue

            # 현재 시점 기준 연체 여부 생성
            # 미완납이지만 5영업일 이상 지연된 송장은 1
            # 이미 완납된 송장은 기존 target 사용
            known_history['known_late_as_of_current'] = np.where(
                known_late_unpaid.loc[known_history.index],
                1.0,
                known_history['target'].astype(float)
            )

            # 같은 고객 + 같은 Allowed_Pay_Days의 과거 연체율 계산
            rate_by_allowed_days = (
                known_history
                .groupby('Allowed_Pay_Days')['known_late_as_of_current']
                .mean()
            )

            # 현재 행의 Allowed_Pay_Days에 매핑
            current_df.loc[current_rows.index, feature] = (
                current_rows['Allowed_Pay_Days']
                .map(rate_by_allowed_days)
                .fillna(0.0)
                .astype(float)
            )

    return current_df


# =========================================
# 6. train_eda_revised 피처 생성
# =========================================

# train은 train 내부의 과거 송장만 사용
train_eda_revised = add_allowed_pay_days_late_rate_past(
    current_df=train_eda_revised,
    history_df=train_eda_revised,
    feature=feature
)


# =========================================
# 7. test_eda_revised 피처 생성
# =========================================

# test는 train 전체 + test 내부 과거 송장을 history로 사용
test_history_revised = pd.concat(
    [train_eda_revised, test_eda_revised],
    axis=0,
    ignore_index=True
)

test_eda_revised = add_allowed_pay_days_late_rate_past(
    current_df=test_eda_revised,
    history_df=test_history_revised,
    feature=feature
)


# =========================================
# 8. 결과 확인
# =========================================

print("train_eda_revised feature describe")
print(train_eda_revised[feature].describe())

print("\ntest_eda_revised feature describe")
print(test_eda_revised[feature].describe())

train_eda_revised feature describe
count    32000.000000
mean         0.059264
std          0.176884
min          0.000000
25%          0.000000
50%          0.007475
75%          0.023474
max          1.000000
Name: cust_allowed_pay_days_late_rate_past, dtype: float64

test_eda_revised feature describe
count    8000.000000
mean        0.054713
std         0.155085
min         0.000000
25%         0.004791
50%         0.007145
75%         0.028070
max         1.000000
Name: cust_allowed_pay_days_late_rate_past, dtype: float64


In [15]:
import numpy as np
import pandas as pd

# =========================================
# 1. 피처명 설정
# =========================================

feature = 'ratio_paid_invoices_late_past'

train_eda_revised[feature] = 0.0
test_eda_revised[feature] = 0.0


# =========================================
# 2. 날짜 컬럼 datetime 변환
# =========================================

date_cols = ['baseline_create_date', 'due_in_date', 'clear_date']

for col in date_cols:
    train_eda_revised[col] = pd.to_datetime(train_eda_revised[col], errors='coerce')
    test_eda_revised[col] = pd.to_datetime(test_eda_revised[col], errors='coerce')


# =========================================
# 3. 영업일 계산 함수
# =========================================

def business_days_between(start_dates, end_dates):
    """
    start_dates부터 end_dates까지의 영업일 수를 계산한다.
    Series와 단일 날짜 모두 처리할 수 있도록 작성했다.
    """
    start_dates = pd.to_datetime(start_dates, errors='coerce')

    # end_dates가 단일 날짜인 경우
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)

    end_dates = pd.to_datetime(end_dates, errors='coerce')

    result = pd.Series(np.nan, index=start_dates.index, dtype='float')

    valid_mask = start_dates.notna() & end_dates.notna()

    if valid_mask.sum() == 0:
        return result

    start_np = start_dates.loc[valid_mask].values.astype('datetime64[D]')
    end_np = end_dates.loc[valid_mask].values.astype('datetime64[D]')

    result.loc[valid_mask] = np.busday_count(start_np, end_np)

    return result


# =========================================
# 4. 타겟 기준 설정
# =========================================

# 타겟 정의와 동일하게 영업일 기준 5일 이상 지연
LATE_BUSINESS_DAYS = 5


# =========================================
# 5. 과거 연체율 생성 함수
# =========================================

def add_ratio_known_late_invoices_past(current_df, history_df, feature):
    """
    고객별 과거 송장 중 현재 시점에서 연체 여부를 알 수 있는 송장을 기준으로
    과거 연체 비율을 계산한다.

    포함되는 송장:
    1. 현재 송장 생성일 이전에 이미 완납된 송장
       - clear_date < 현재 baseline_create_date
       - clear_date와 due_in_date 기준으로 5영업일 이상 지연 여부를 계산한다.

    2. 현재 송장 생성일 기준 아직 미완납이지만 이미 5영업일 이상 지연된 송장
       - clear_date가 없거나 clear_date >= 현재 baseline_create_date
       - 현재 baseline_create_date 기준 due_in_date로부터 5영업일 이상 지남
       - 현재 시점에서 이미 연체가 확정되었으므로 1로 반영한다.

    제외되는 송장:
    1. 아직 완납되지 않았고 5영업일 이상 지연도 확정되지 않은 송장
       - 정상인지 연체인지 현재 시점에서 알 수 없으므로 제외한다.
    """

    for cust, current_rows_for_customer in current_df.groupby('cust_number', sort=False):

        # 같은 고객의 전체 이력
        customer_history = history_df[history_df['cust_number'].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):

            # 현재 송장 생성일보다 과거에 생성된 송장만 사용
            prior = customer_history[
                customer_history['baseline_create_date'] < current_date
            ].copy()

            if prior.empty:
                continue

            # 1. 현재 시점 이전에 이미 완납된 송장
            paid_before_current = (
                prior['clear_date'].notna()
                & (prior['clear_date'] < current_date)
            )

            # 2. 현재 시점 기준 아직 미완납인 송장
            unpaid_as_of_current = (
                prior['clear_date'].isna()
                | (prior['clear_date'] >= current_date)
            )

            # due_in_date부터 현재 송장 생성일까지 지난 영업일 수
            business_days_past_due = business_days_between(
                prior['due_in_date'],
                current_date
            )

            # 3. 미완납이지만 현재 시점에서 이미 5영업일 이상 지연 확정된 송장
            known_late_unpaid = (
                unpaid_as_of_current
                & (business_days_past_due >= LATE_BUSINESS_DAYS)
            )

            # 현재 시점에서 연체 여부를 알 수 있는 송장만 사용
            known_history = prior[
                paid_before_current | known_late_unpaid
            ].copy()

            if known_history.empty:
                continue

            # 완납된 송장의 실제 지연 여부 계산
            paid_late_days = business_days_between(
                known_history['due_in_date'],
                known_history['clear_date']
            )

            # 현재 시점 기준 연체 여부 생성
            # 미완납이지만 이미 5영업일 이상 지연된 송장은 1
            # 완납된 송장은 due_in_date ~ clear_date 기준으로 5영업일 이상 지연 여부 계산
            known_history['known_late_as_of_current'] = np.where(
                known_late_unpaid.loc[known_history.index],
                1.0,
                (paid_late_days >= LATE_BUSINESS_DAYS).astype(float)
            )

            # 현재 시점에서 알 수 있는 과거 송장 중 연체 비율
            current_late_rate = known_history['known_late_as_of_current'].mean()

            # 같은 baseline_create_date를 가진 현재 행들에 동일하게 입력
            current_df.loc[current_rows.index, feature] = float(current_late_rate)

    return current_df


# =========================================
# 6. train 데이터 피처 생성
# =========================================

train_eda_revised = add_ratio_known_late_invoices_past(
    current_df=train_eda_revised,
    history_df=train_eda_revised,
    feature=feature
)


# =========================================
# 7. test 데이터 피처 생성
# =========================================

# test는 train 전체 + test 내부 과거 송장을 history로 사용
test_history_revised = pd.concat(
    [train_eda_revised, test_eda_revised],
    axis=0,
    ignore_index=True
)

test_eda_revised = add_ratio_known_late_invoices_past(
    current_df=test_eda_revised,
    history_df=test_history_revised,
    feature=feature
)


# =========================================
# 8. 결과 확인
# =========================================

print("train_eda_revised feature describe")
print(train_eda_revised[feature].describe())

print("\ntest_eda_revised feature describe")
print(test_eda_revised[feature].describe())

train_eda_revised feature describe
count    32000.000000
mean         0.074160
std          0.191275
min          0.000000
25%          0.000000
50%          0.009333
75%          0.036145
max          1.000000
Name: ratio_paid_invoices_late_past, dtype: float64

test_eda_revised feature describe
count    8000.000000
mean        0.068153
std         0.169745
min         0.000000
25%         0.008432
50%         0.012195
75%         0.042254
max         1.000000
Name: ratio_paid_invoices_late_past, dtype: float64


In [17]:
import numpy as np
import pandas as pd

# =========================================
# 1. 피처명 설정
# =========================================

feature = 'avg_days_late_paid_late_past'

train_eda_revised[feature] = 0.0
test_eda_revised[feature] = 0.0


# =========================================
# 2. 날짜 컬럼 datetime 변환
# =========================================

date_cols = ['baseline_create_date', 'due_in_date', 'clear_date']

for col in date_cols:
    train_eda_revised[col] = pd.to_datetime(train_eda_revised[col], errors='coerce')
    test_eda_revised[col] = pd.to_datetime(test_eda_revised[col], errors='coerce')


# =========================================
# 3. 영업일 계산 함수
# =========================================

def business_days_between(start_dates, end_dates):
    """
    start_dates부터 end_dates까지의 영업일 수를 계산한다.
    Series와 단일 날짜를 모두 처리할 수 있도록 작성한 함수이다.
    """

    start_dates = pd.to_datetime(start_dates, errors='coerce')

    # end_dates가 단일 날짜인 경우 Series로 변환
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)

    end_dates = pd.to_datetime(end_dates, errors='coerce')

    result = pd.Series(np.nan, index=start_dates.index, dtype='float')

    valid_mask = start_dates.notna() & end_dates.notna()

    if valid_mask.sum() == 0:
        return result

    start_np = start_dates.loc[valid_mask].values.astype('datetime64[D]')
    end_np = end_dates.loc[valid_mask].values.astype('datetime64[D]')

    result.loc[valid_mask] = np.busday_count(start_np, end_np)

    return result


# =========================================
# 4. 타겟 기준 설정
# =========================================

# 타겟 정의와 동일하게 영업일 기준 5일 이상 지연
LATE_BUSINESS_DAYS = 5


# =========================================
# 5. 평균 지연일 생성 함수
# =========================================

def add_avg_days_late_known_late_past(current_df, history_df, feature):
    """
    고객별 과거 송장 중 현재 시점에서 이미 연체가 확인된 송장만 대상으로
    평균 영업일 지연일을 계산한다.

    포함되는 송장:
    1. 현재 송장 생성일 이전에 이미 완납되었고,
       due_in_date ~ clear_date 기준으로 5영업일 이상 지연된 송장

    2. 현재 송장 생성일 기준 아직 미완납이지만,
       due_in_date ~ 현재 baseline_create_date 기준으로 5영업일 이상 지연된 송장

    제외되는 송장:
    1. 아직 완납되지 않았고 5영업일 이상 지연도 확정되지 않은 송장
    2. 완납되었지만 5영업일 이상 지연되지 않은 정상 송장
    """

    for cust, current_rows_for_customer in current_df.groupby('cust_number', sort=False):

        # 같은 고객의 전체 이력
        customer_history = history_df[history_df['cust_number'].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):

            # 현재 송장 생성일보다 과거에 생성된 송장만 사용
            prior = customer_history[
                customer_history['baseline_create_date'] < current_date
            ].copy()

            if prior.empty:
                continue

            # 1. 현재 시점 이전에 이미 완납된 송장
            paid_before_current = (
                prior['clear_date'].notna()
                & (prior['clear_date'] < current_date)
            )

            # 완납된 송장의 실제 영업일 지연일 계산
            paid_late_days = business_days_between(
                prior['due_in_date'],
                prior['clear_date']
            )

            # 완납된 송장 중 5영업일 이상 지연된 송장
            paid_late = (
                paid_before_current
                & (paid_late_days >= LATE_BUSINESS_DAYS)
            )

            # 2. 현재 시점 기준 아직 미완납인 송장
            unpaid_as_of_current = (
                prior['clear_date'].isna()
                | (prior['clear_date'] >= current_date)
            )

            # 미완납 송장이 현재 시점 기준으로 만기일 이후 몇 영업일 지났는지 계산
            unpaid_late_days_as_of_current = business_days_between(
                prior['due_in_date'],
                current_date
            )

            # 미완납이지만 현재 시점에서 이미 5영업일 이상 지연 확정된 송장
            known_late_unpaid = (
                unpaid_as_of_current
                & (unpaid_late_days_as_of_current >= LATE_BUSINESS_DAYS)
            )

            # 현재 시점에서 이미 연체가 확인된 송장만 선택
            known_late_history = prior[
                paid_late | known_late_unpaid
            ].copy()

            if known_late_history.empty:
                continue

            # 각 과거 송장별 현재 시점 기준 지연일 생성
            # 완납된 연체 송장: due_in_date ~ clear_date 기준 지연일
            # 미완납 연체 송장: due_in_date ~ 현재 baseline_create_date 기준 지연일
            known_late_history['days_late_as_of_current'] = np.where(
                known_late_unpaid.loc[known_late_history.index],
                unpaid_late_days_as_of_current.loc[known_late_history.index],
                paid_late_days.loc[known_late_history.index]
            )

            # 현재 시점에서 확인 가능한 연체 송장들의 평균 지연일
            avg_late_days = known_late_history['days_late_as_of_current'].mean()

            current_df.loc[current_rows.index, feature] = float(avg_late_days)

    return current_df


# =========================================
# 6. train 데이터 피처 생성
# =========================================

# train은 train 내부의 과거 송장만 사용
train_eda_revised = add_avg_days_late_known_late_past(
    current_df=train_eda_revised,
    history_df=train_eda_revised,
    feature=feature
)


# =========================================
# 7. test 데이터 피처 생성
# =========================================

# test는 train 전체 + test 내부 과거 송장을 history로 사용
test_history_revised = pd.concat(
    [train_eda_revised, test_eda_revised],
    axis=0,
    ignore_index=True
)

test_eda_revised = add_avg_days_late_known_late_past(
    current_df=test_eda_revised,
    history_df=test_history_revised,
    feature=feature
)


# =========================================
# 8. 결과 확인
# =========================================

print("train_eda_revised feature describe")
print(train_eda_revised[feature].describe())

print("\ntest_eda_revised feature describe")
print(test_eda_revised[feature].describe())

train_eda_revised feature describe
count    32000.000000
mean        13.335156
std         14.151372
min          0.000000
25%          0.000000
50%          8.000000
75%         20.224359
max         86.000000
Name: avg_days_late_paid_late_past, dtype: float64

test_eda_revised feature describe
count    8000.000000
mean       17.649019
std        13.684429
min         0.000000
25%         6.705882
50%        12.489130
75%        32.066667
max        86.000000
Name: avg_days_late_paid_late_past, dtype: float64


In [18]:
import numpy as np
import pandas as pd

# =========================================
# 1. 피처명 설정
# =========================================

feature = 'recent_5_late_rate'

train_eda_revised[feature] = 0.0
test_eda_revised[feature] = 0.0


# =========================================
# 2. 날짜 컬럼 datetime 변환
# =========================================

date_cols = ['baseline_create_date', 'due_in_date', 'clear_date']

for col in date_cols:
    train_eda_revised[col] = pd.to_datetime(train_eda_revised[col], errors='coerce')
    test_eda_revised[col] = pd.to_datetime(test_eda_revised[col], errors='coerce')


# =========================================
# 3. 영업일 계산 함수
# =========================================

def business_days_between(start_dates, end_dates):
    """
    start_dates부터 end_dates까지의 영업일 수를 계산하는 함수이다.
    Series와 단일 날짜를 모두 처리할 수 있다.
    """

    start_dates = pd.to_datetime(start_dates, errors='coerce')

    # end_dates가 단일 날짜인 경우 Series로 변환한다.
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)

    end_dates = pd.to_datetime(end_dates, errors='coerce')

    result = pd.Series(np.nan, index=start_dates.index, dtype='float')

    valid_mask = start_dates.notna() & end_dates.notna()

    if valid_mask.sum() == 0:
        return result

    start_np = start_dates.loc[valid_mask].values.astype('datetime64[D]')
    end_np = end_dates.loc[valid_mask].values.astype('datetime64[D]')

    result.loc[valid_mask] = np.busday_count(start_np, end_np)

    return result


def add_business_days(start_dates, n_business_days):
    """
    start_dates에서 n_business_days만큼 영업일을 더한 날짜를 계산하는 함수이다.
    미완납 송장이 언제부터 '연체 확정 이력'으로 관찰 가능한지 계산할 때 사용한다.
    """

    start_dates = pd.to_datetime(start_dates, errors='coerce')

    result = pd.Series(pd.NaT, index=start_dates.index)

    valid_mask = start_dates.notna()

    if valid_mask.sum() == 0:
        return result

    start_np = start_dates.loc[valid_mask].values.astype('datetime64[D]')

    result.loc[valid_mask] = pd.to_datetime(
        np.busday_offset(
            start_np,
            n_business_days,
            roll='forward'
        )
    )

    return result


# =========================================
# 4. 기준 설정
# =========================================

# 타겟 정의와 동일하게 영업일 기준 5일 이상 지연
LATE_BUSINESS_DAYS = 5

# 최근 몇 건을 볼 것인지 설정
N_RECENT = 5


# =========================================
# 5. 최근 5건 기준 연체율 생성 함수
# =========================================

def add_recent_5_late_rate(current_df, history_df, feature):
    """
    고객별로 현재 시점에서 연체 여부를 알 수 있는 과거 송장 중
    최근 5건의 연체 비율을 계산한다.

    포함되는 과거 송장:
    1. 현재 송장 생성일 이전에 이미 완납된 송장
       - clear_date < 현재 baseline_create_date
       - clear_date와 due_in_date를 기준으로 실제 연체 여부를 계산한다.

    2. 현재 송장 생성일 기준 아직 미완납이지만 이미 5영업일 이상 지연된 송장
       - clear_date가 없거나 clear_date >= 현재 baseline_create_date
       - 현재 baseline_create_date 기준 due_in_date로부터 5영업일 이상 지남
       - 현재 시점에서 이미 연체가 확정되었으므로 연체 1로 반영한다.

    제외되는 과거 송장:
    1. 아직 미완납이고 5영업일 이상 지연도 확정되지 않은 송장
       - 현재 시점에서 정상인지 연체인지 알 수 없으므로 제외한다.
    """

    for cust, current_rows_for_customer in current_df.groupby('cust_number', sort=False):

        # 같은 고객의 전체 이력
        customer_history = history_df[history_df['cust_number'].eq(cust)].copy()

        for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):

            # 현재 송장 생성일보다 과거에 생성된 송장만 사용한다.
            prior = customer_history[
                customer_history['baseline_create_date'] < current_date
            ].copy()

            if prior.empty:
                continue

            # 1. 현재 시점 이전에 이미 완납된 송장
            paid_before_current = (
                prior['clear_date'].notna()
                & (prior['clear_date'] < current_date)
            )

            # 2. 현재 시점 기준 아직 미완납인 송장
            unpaid_as_of_current = (
                prior['clear_date'].isna()
                | (prior['clear_date'] >= current_date)
            )

            # 미완납 송장이 현재 시점 기준으로 만기일 이후 몇 영업일 지났는지 계산한다.
            unpaid_late_days_as_of_current = business_days_between(
                prior['due_in_date'],
                current_date
            )

            # 3. 미완납이지만 현재 시점에서 이미 5영업일 이상 지연 확정된 송장
            known_late_unpaid = (
                unpaid_as_of_current
                & (unpaid_late_days_as_of_current >= LATE_BUSINESS_DAYS)
            )

            # 현재 시점에서 연체 여부를 알 수 있는 송장만 사용한다.
            known_history = prior[
                paid_before_current | known_late_unpaid
            ].copy()

            if known_history.empty:
                continue

            # 완납된 송장의 실제 영업일 지연일 계산
            paid_late_days = business_days_between(
                known_history['due_in_date'],
                known_history['clear_date']
            )

            # 미완납 연체 확정 여부를 known_history 인덱스에 맞춘다.
            known_late_unpaid_for_known_history = (
                known_late_unpaid
                .reindex(known_history.index)
                .fillna(False)
            )

            # 현재 시점 기준 연체 여부 생성
            # 미완납이지만 5영업일 이상 지연된 송장은 1
            # 완납된 송장은 due_in_date ~ clear_date 기준으로 5영업일 이상 지연 여부 계산
            known_history['known_late_as_of_current'] = np.where(
                known_late_unpaid_for_known_history,
                1.0,
                (paid_late_days >= LATE_BUSINESS_DAYS).astype(float)
            )

            # 정렬 기준 날짜 생성
            # 완납 송장: clear_date에 연체 여부를 알 수 있다.
            # 미완납 연체 확정 송장: due_in_date + 5영업일 시점부터 연체 이력으로 볼 수 있다.
            known_history['_known_event_date'] = known_history['clear_date']

            known_history.loc[
                known_late_unpaid_for_known_history,
                '_known_event_date'
            ] = add_business_days(
                known_history.loc[known_late_unpaid_for_known_history, 'due_in_date'],
                LATE_BUSINESS_DAYS
            )

            # 최근 5건 선택
            recent_known = (
                known_history
                .sort_values(
                    ['_known_event_date', 'baseline_create_date', 'due_in_date', 'clear_date']
                )
                .tail(N_RECENT)
            )

            # 최근 5건 중 연체 비율 계산
            current_late_rate = recent_known['known_late_as_of_current'].mean()

            current_df.loc[current_rows.index, feature] = float(current_late_rate)

    return current_df


# =========================================
# 6. train 데이터 피처 생성
# =========================================

# train은 train 내부의 과거 송장만 사용한다.
train_eda_revised = add_recent_5_late_rate(
    current_df=train_eda_revised,
    history_df=train_eda_revised,
    feature=feature
)


# =========================================
# 7. test 데이터 피처 생성
# =========================================

# test는 train 전체 + test 내부 과거 송장을 history로 사용한다.
test_history_revised = pd.concat(
    [train_eda_revised, test_eda_revised],
    axis=0,
    ignore_index=True
)

test_eda_revised = add_recent_5_late_rate(
    current_df=test_eda_revised,
    history_df=test_history_revised,
    feature=feature
)


# =========================================
# 8. 결과 확인
# =========================================

print("train_eda_revised feature describe")
print(train_eda_revised[feature].describe())

print("\ntest_eda_revised feature describe")
print(test_eda_revised[feature].describe())

train_eda_revised feature describe
count    32000.000000
mean         0.097278
std          0.225117
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: recent_5_late_rate, dtype: float64

test_eda_revised feature describe
count    8000.000000
mean        0.082154
std         0.208388
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: recent_5_late_rate, dtype: float64


In [ ]:
import numpy as np
import pandas as pd

# =========================================
# 1. 피처명 설정
# =========================================

feature = 'sum_outstanding_amount_past'

train_eda_revised[feature] = 0.0
test_eda_revised[feature] = 0.0


# =========================================
# 2. 날짜 컬럼 datetime 변환
# =========================================

date_cols = ['baseline_create_date', 'due_in_date', 'clear_date']

for col in date_cols:
    train_eda_revised[col] = pd.to_datetime(train_eda_revised[col], errors='coerce')
    test_eda_revised[col] = pd.to_datetime(test_eda_revised[col], errors='coerce')


# =========================================
# 3. 현재 시점 기준 미완납 과거 송장 총금액 생성 함수
# =========================================

def add_sum_outstanding_amount_past(current_df, history_df, feature):
    """
    고객별로 현재 송장 생성일 기준 아직 결제되지 않은 과거 송장의 총금액을 계산한다.

    포함되는 송장:
    1. 현재 송장보다 과거에 생성된 송장
       - baseline_create_date < 현재 baseline_create_date

    2. 현재 시점 기준 아직 미완납인 송장
       - clear_date가 결측이거나
       - clear_date >= 현재 baseline_create_date

    의미:
    현재 송장 생성 시점에서 이 고객이 아직 갚지 않은 과거 송장 금액의 합이다.
    """

    for cust, current_rows_for_customer in current_df.groupby('cust_number', sort=False):

        # 같은 고객의 전체 이력
        customer_history = history_df[history_df['cust_number'].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):

            # 현재 시점 기준 아직 결제되지 않은 과거 송장
            outstanding = customer_history[
                (customer_history['baseline_create_date'] < current_date)
                & (
                    customer_history['clear_date'].isna()
                    | (customer_history['clear_date'] >= current_date)
                )
            ]

            # 미결제 과거 송장 총금액
            outstanding_amount = outstanding['amount_in_usd'].sum()

            # 음수 방지
            current_df.loc[current_rows.index, feature] = max(
                float(outstanding_amount),
                0.0
            )

    return current_df


# =========================================
# 4. train 데이터 피처 생성
# =========================================

# train은 train 내부의 과거 송장만 사용
train_eda_revised = add_sum_outstanding_amount_past(
    current_df=train_eda_revised,
    history_df=train_eda_revised,
    feature=feature
)


# =========================================
# 5. test 데이터 피처 생성
# =========================================

# test는 train 전체 + test 내부 과거 송장을 history로 사용
test_history_revised = pd.concat(
    [train_eda_revised, test_eda_revised],
    axis=0,
    ignore_index=True
)

test_eda_revised = add_sum_outstanding_amount_past(
    current_df=test_eda_revised,
    history_df=test_history_revised,
    feature=feature
)


# =========================================
# 6. 결과 확인
# =========================================

print("train_eda_revised feature describe")
print(train _eda_revised[feature].describe())

print("\ntest_eda_revised feature describe")
print(test_eda_revised[feature].describe())

train_eda_revised feature describe
count    32000.000000
mean       740.875133
std       1071.324304
min          0.000000
25%         35.262890
50%        162.661170
75%        650.793740
max       3635.495781
Name: sum_outstanding_amount_past, dtype: float64

test_eda_revised feature describe
count    8000.000000
mean      743.520660
std      1100.702257
min         0.000000
25%        26.103770
50%       138.836643
75%       754.917854
max      4120.210512
Name: sum_outstanding_amount_past, dtype: float64


In [21]:
train_eda_revised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 25 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   business_code                         32000 non-null  object        
 1   cust_number                           32000 non-null  object        
 2   name_customer                         32000 non-null  object        
 3   clear_date                            32000 non-null  datetime64[ns]
 4   buisness_year                         32000 non-null  float64       
 5   due_in_date                           32000 non-null  datetime64[ns]
 6   posting_id                            32000 non-null  float64       
 7   baseline_create_date                  32000 non-null  datetime64[ns]
 8   cust_payment_terms                    32000 non-null  object        
 9   target_old                            32000 non-null  int64         
 10

In [ ]:
train_eda_revised.to_csv('../data/train_eda_revised.csv', index=False)
test_eda_revised.to_csv('../data/test_eda_revised.csv', index=False)
213